# Phase 5 screen A — the training schedule, on fold 0 only

Phase 4 run 1 is `efficientnet_b0`, **1 epoch**, flat Adam 1e-4, no AMP: **435 optimizer steps**.
It reaches 0.7779 OOF against pseudo-labels that themselves score 0.862 gold, i.e. it cannot fit
the signal it was given. The 0.934 public notebook trains **10 epochs** with OneCycle and AMP.
Epoch count has never been moved in this project.

**Why one fold and not five.** A 5-fold run costs 35 min per config, which buys about six configs a
session and forces the ordering to be guessed in advance. Fold 0 alone costs ~6.5 min at run 1's
settings, so the ordering can be measured instead. Fold 0 ranks candidates; it **never promotes
one** — the promotion rule stays a 5-fold pooled OOF paired delta.

**Decisions are read off the six stable labels**, those at >= 0.10 corpus prevalence: Medial
Meniscus, Lateral Meniscus, Effusion, Synovitis, Baker's, Contusion. Full 12-label macro is printed
alongside but not decided on. Two reasons, and the second is correctness rather than noise:
Fracture contributes ~12 positives on fold 0 and MCL ~17, so the full macro weights those equally
with Medial Meniscus's ~278; and in bootstrap replicates where Fracture loses a class,
`per_label_auc` returns NaN for it, so `nanmean` averages 11 labels in some replicates and 12 in
others — a CI over two different metrics. The arrays are therefore **sliced to the six columns
before** `paired_macro_auc_delta` sees them, because it macro-averages over every column it is
handed.

**Every config also scores a held-in training subsample of the same size as the val fold.** That
train-vs-val gap across the epoch ladder is the real overfit detector, and it comes free.

In [ ]:
import glob, hashlib, os, shutil, sys, time

GIT_SHA = 'phase5-screen-a'
FOLDS_PRIMARY_V2_SHA256 = 'cace8c45733ee7920a442fa4ae3f1db4a0d76401504e4729b49562d5eab52047'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
assert len(PREPPED_DIRS) == 4, f'expected 4 prep-shard outputs, found {len(PREPPED_DIRS)}'

uid_to_npz = {}
for d in PREPPED_DIRS:
    for p in sorted(glob.glob(os.path.join(d, '*.npz'))):
        uid = os.path.splitext(os.path.basename(p))[0]
        assert uid not in uid_to_npz, f'duplicate artifact for {uid}'
        uid_to_npz[uid] = p
print(f'{len(uid_to_npz)} prepped artifacts mounted')

NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')
print('src/knee mounted from', SRC)

In [ ]:
import torch

assert torch.cuda.is_available(), 'no GPU attached'
for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, sm_{major}{minor}')
    assert (major, minor) >= (7, 0), f'GPU {i} is sm_{major}{minor} (P100?) -- need T4'
device = 'cuda'

# train_one_epoch grew a scaler/scheduler path in Phase 5; fail here rather than
# silently screening the old recipe under new config names.
import inspect
from knee.train import train_one_epoch as _t
from knee.dataset import PreppedStudyDataset as _d
assert {'scaler', 'scheduler'} <= set(inspect.signature(_t).parameters), \
    'src dataset predates the AMP/scheduler support this kernel screens'
assert 'sides' in inspect.signature(_d.__init__).parameters, \
    'src dataset predates the laterality side override this kernel screens'
print('src carries the Phase 5 train_one_epoch and side override')

In [ ]:
import numpy as np
import pandas as pd

from knee.infer import LABEL_COLUMNS
from knee.train import (Timer, evaluate, load_gold_holdout, log_experiment,
                        make_folds, train_one_epoch, train_val_split)
from knee.dataset import PreppedStudyDataset
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.model import KneeModel

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

pseudo_path = glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0]
pseudo = pd.read_csv(pseudo_path)
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]].copy()
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS
assert len(labels_all) == 4407 and labels_all['StudyInstanceUID'].is_unique
_vals = labels_all[LABEL_COLUMNS].to_numpy(dtype=float)
assert np.isfinite(_vals).all() and (_vals >= 0).all() and (_vals <= 1).all()
labels_all['StudyInstanceUID'] = labels_all['StudyInstanceUID'].astype(str)

# soft scores train; binarized scores are what AUC can be computed against
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (_vals >= 0.5).astype(float)

folds = make_folds(all_uids, n_folds=5, seed=0)
_digest = hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()).hexdigest()
assert _digest == FOLDS_PRIMARY_V2_SHA256, (
    f'fold assignment is not primary_v2 ({_digest}) -- every paired delta against '
    'run 1 would be meaningless')
print('fold assignment verified against primary_v2')

gold_df = train_df[train_df['ACL'].notna()].reset_index(drop=True)
assert len(gold_df) == 58
gold_df[['StudyInstanceUID']].to_csv('gold_tmp.csv', index=False)
holdout = load_gold_holdout('gold_tmp.csv')

VAL_FOLD = 0
train_uids, val_uids = train_val_split(folds, val_fold=VAL_FOLD, exclude_uids=holdout)
assert not (set(train_uids) & set(val_uids))
assert not (set(train_uids) | set(val_uids)) & holdout
print(f'fold {VAL_FOLD}: {len(train_uids)} train / {len(val_uids)} val, gold excluded from both')

## The baseline arm

Run 1's own out-of-fold predictions, pulled from the `knee-phase4-train` kernel output. Every
paired delta below is against these — the same studies, the same targets, so the pairing cancels
study-level variance. Without it a single fold cannot resolve the differences being screened.

In [ ]:
oof_dir = os.path.dirname(glob.glob('/kaggle/input/**/oof_pred.npy', recursive=True)[0])
base_pred = np.load(f'{oof_dir}/oof_pred.npy')
base_true = np.load(f'{oof_dir}/oof_true.npy')
base_uids = np.load(f'{oof_dir}/oof_uids.npy', allow_pickle=True).astype(str)
print('run 1 OOF:', base_pred.shape)

sel = np.array([folds[u] == VAL_FOLD for u in base_uids])
base_fold_uids = base_uids[sel]
# Order must match val_uids exactly or the pairing silently compares different studies
assert list(base_fold_uids) == list(val_uids), 'run-1 OOF fold-0 rows do not line up with val_uids'
BASE_PRED = base_pred[sel]
BASE_TRUE = base_true[sel]
print(f'baseline arm: {BASE_PRED.shape[0]} fold-{VAL_FOLD} studies')

STABLE = ["Medial Meniscus", "Lateral Meniscus", "Effusion", "Synovitis", "Baker's", "Contusion"]
SIX = [LABEL_COLUMNS.index(l) for l in STABLE]
print('stable-label columns:', dict(zip(STABLE, SIX)))
print(f'run 1 fold-{VAL_FOLD}: full macro {macro_auc(BASE_TRUE, BASE_PRED):.4f}, '
      f'stable-6 {macro_auc(BASE_TRUE[:, SIX], BASE_PRED[:, SIX]):.4f}')

# A held-in subsample the same size as the val fold, so train and val macros are
# comparable numbers rather than one over 3.5k studies and one over 0.9k.
rng = np.random.default_rng(0)
TRAIN_PROBE = sorted(rng.choice(train_uids, size=len(val_uids), replace=False).tolist())

# --- corrected laterality, from the phase5-laterality kernel output ----------
# Same rule resolve_study_laterality now applies: a real tag wins, geometry only
# where no tag resolved. The count asserts pin this against drifting from the
# resolver -- if they fire, the two have diverged and the map is not what the
# submission path would produce for the same study.
lat = pd.read_csv(glob.glob('/kaggle/input/**/laterality_geometry_check.csv', recursive=True)[0])
def _final(r):
    if r['tag_side'] in ('L', 'R'):
        return r['tag_side']
    if r['geom_side'] in ('L', 'R'):
        return r['geom_side']
    return None
lat['final'] = lat.apply(_final, axis=1)
SIDES = {str(u): s for u, s in zip(lat['StudyInstanceUID'], lat['final'])
         if s in ('L', 'R')}
assert len(lat) == 4407
assert len(SIDES) == 4341, f'expected 4341 resolved sides, got {len(SIDES)}'
_from_tag = int(lat['tag_side'].isin(['L', 'R']).sum())
assert _from_tag == 2252, _from_tag
print(f'corrected sides: {len(SIDES)}/4407 = {len(SIDES)/4407:.1%} '
      f'(was {_from_tag/4407:.1%} on tags alone)')

In [ ]:
MAX_SERIES, N_SLICES, BATCH_SIZE = 1, 16, 8
EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = ('date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,'
           'medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,'
           'pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,'
           'macro_auc,paired_delta,train_minutes,inference_seconds,promoted')
with open(EXPERIMENTS_CSV, 'w') as f:
    f.write(_HEADER + '\n')

def loader_for(uids, labels_df, shuffle=False):
    ds = PreppedStudyDataset(uids, NPZ_ROOT, labels_df=labels_df,
                             n_slices=N_SLICES, max_series=MAX_SERIES)
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

def loader_for(uids, labels_df, shuffle=False, sides=None):
    ds = PreppedStudyDataset(uids, NPZ_ROOT, labels_df=labels_df,
                             n_slices=N_SLICES, max_series=MAX_SERIES, sides=sides)
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = loader_for(train_uids, labels_all, shuffle=True)
val_loader   = loader_for(val_uids, labels_eval)
probe_loader = loader_for(TRAIN_PROBE, labels_eval)

# same studies, same targets, only the mirroring changes
train_loader_lat = loader_for(train_uids, labels_all, shuffle=True, sides=SIDES)
val_loader_lat   = loader_for(val_uids, labels_eval, sides=SIDES)
probe_loader_lat = loader_for(TRAIN_PROBE, labels_eval, sides=SIDES)

# Cheapest first and one CSV row appended per config: if the session hits the GPU
# cap mid-grid, the rows already written still rank what ran.
CONFIGS = [
    dict(name='s1_e1_flat_amp',   epochs=1,  onecycle=False, amp=True),
    dict(name='s2_e4_onecycle_amp',  epochs=4,  onecycle=True,  amp=True),
    dict(name='s3_e8_onecycle_amp',  epochs=8,  onecycle=True,  amp=True),
    dict(name='s4_e8_flat_amp',      epochs=8,  onecycle=False, amp=True),
    dict(name='s5_e12_onecycle_amp', epochs=12, onecycle=True,  amp=True),
    # laterality at run 1's schedule: isolates the mirroring against s1, which is
    # the same recipe on the stored sides
    dict(name='s6_e1_flat_amp_lat',   epochs=1,  onecycle=False, amp=True, lat=True),
]
LR = 1e-4
print(f'{len(CONFIGS)} configs on fold {VAL_FOLD}')

### What each config isolates

- **s1** is run 1's recipe with AMP and nothing else, so any difference from the baseline arm is
  AMP's arithmetic. It is also the harness check: it should land near run 1's own fold-0 macro.
- **s2, s3, s5** walk the epoch ladder 4 / 8 / 12 under OneCycle. Coarse on purpose — the backbone
  is going to change, and a schedule tuned finely against `efficientnet_b0` would not survive it.
- **s4** is s3 with a flat learning rate, which separates *more epochs* from *the schedule*. Without
  it the epoch ladder confounds the two.

In [ ]:
from tqdm.auto import tqdm

rows = []

def run_config(cfg):
    """Train one config on fold 0, score val + a held-in probe of equal size,
    append one experiments.csv row, and return the summary dict. A function
    rather than an inlined loop body so the combined config chosen after the
    ladder runs through exactly this code path and not a copy of it."""
    torch.manual_seed(0)
    model = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS),
                      pretrained=True).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg['amp']) if cfg['amp'] else None
    sched = None
    if cfg['onecycle']:
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=LR, epochs=cfg['epochs'], steps_per_epoch=len(train_loader))

    use_lat = cfg.get('lat', False)
    tl = train_loader_lat if use_lat else train_loader
    vl = val_loader_lat if use_lat else val_loader
    pl = probe_loader_lat if use_lat else probe_loader

    timer = Timer()
    for ep in range(cfg['epochs']):
        with timer:
            loss = train_one_epoch(model, tl, opt, device=device,
                                   scaler=scaler, scheduler=sched)
        print(f"  {cfg['name']} epoch {ep+1}/{cfg['epochs']} loss {loss:.4f}")

    t0 = time.time()
    y_true, y_pred = evaluate(model, vl, device=device)
    infer_s = time.time() - t0
    probe_true, probe_pred = evaluate(model, pl, device=device)

    full = macro_auc(y_true, y_pred)
    six = macro_auc(y_true[:, SIX], y_pred[:, SIX])
    six_train = macro_auc(probe_true[:, SIX], probe_pred[:, SIX])
    delta, lo, hi = paired_macro_auc_delta(y_true[:, SIX], BASE_PRED[:, SIX], y_pred[:, SIX])

    print(f"{cfg['name']}: full {full:.4f} | stable-6 val {six:.4f} train {six_train:.4f} "
          f"(gap {six_train - six:+.4f}) | delta vs run1 {delta:+.4f} [{lo:+.4f}, {hi:+.4f}] "
          f"| {timer.minutes:.1f} min")

    aucs = per_label_auc(y_true, y_pred)
    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA,
                   config_hash=f"screen_{cfg['name']}",
                   hypothesis=('Phase 5 screen A (fold 0 only, ranks candidates, promotes none): '
                               f"epochs={cfg['epochs']} onecycle={cfg['onecycle']} amp={cfg['amp']} "
                               f"laterality_v2={use_lat}, "
                               'otherwise run 1 (efficientnet_b0, 1x16, Adam 1e-4, batch 8)'),
                   fold_set=f'primary_v2_fold{VAL_FOLD}', seed=0,
                   per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                   macro_auc=float(full), paired_delta=float(delta),
                   train_minutes=timer.minutes, inference_seconds=infer_s, promoted=False)

    np.save(f"/kaggle/working/pred_{cfg['name']}.npy", y_pred)
    rows.append(dict(name=cfg['name'], lat=use_lat, full=full, six=six, six_train=six_train,
                     gap=six_train - six, delta=delta, lo=lo, hi=hi,
                     minutes=timer.minutes, infer_s=infer_s))
    pd.DataFrame(rows).to_csv('/kaggle/working/screen_summary.csv', index=False)

    del model, opt, scaler, sched
    torch.cuda.empty_cache()
    return rows[-1]


for cfg in CONFIGS:
    run_config(cfg)

## Reading the screen

A config is a candidate for the 5-fold confirmation only if its paired CI excludes zero. The
train-vs-val gap column is the overfit detector: if it widens sharply as epochs rise while the val
column stops moving, the next lever is augmentation and regularisation, not more epochs.

In [ ]:
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print()

# The best schedule is not known until the ladder has run, so the combined
# config is built here rather than pre-declared. One extra run beats a second
# kernel round trip, and NOTES already names "a kernel that finished and sat
# unread" as this project's process failure.
ladder = summary[~summary['lat']]
best_sched = ladder.sort_values('six', ascending=False).iloc[0]
print(f"best schedule on the ladder: {best_sched['name']} (stable-6 {best_sched['six']:.4f})")
combo = next(c for c in CONFIGS if c['name'] == best_sched['name'])
if combo['epochs'] > 1:
    run_config(dict(combo, name=f"s7_{combo['name']}_lat", lat=True))
    summary = pd.DataFrame(rows)
    print(summary.to_string(index=False))
else:
    print('best schedule is the 1-epoch config; s6 already covers it with laterality')
print()
wins = summary[summary['lo'] > 0]
if len(wins):
    best = wins.sort_values('six', ascending=False).iloc[0]
    print(f"best candidate: {best['name']}  stable-6 {best['six']:.4f}  "
          f"delta {best['delta']:+.4f} [{best['lo']:+.4f}, {best['hi']:+.4f}]")
    print('-> confirm on 5 folds before promoting anything')
else:
    print('no config clears the paired CI on fold 0 -- do not spend a 5-fold run')